# ML02：经典模型、不平衡指标与数据泄漏

比较合理 baselines，严格区分 model selection、threshold selection 与 test evaluation。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import sklearn
print('sklearn:', sklearn.__version__)


## 2. 固定 train/validation/test 与候选模型
合成 tabular classification，不是机器人 rollout。所有 preprocessing 都放在 Pipeline 内。


In [ ]:
from time import perf_counter
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
    precision_score, recall_score, accuracy_score, brier_score_loss,
    confusion_matrix, precision_recall_curve, roc_curve)
from sklearn.calibration import calibration_curve
X, y = make_classification(n_samples=1200, n_features=12, n_informative=6,
    n_redundant=2, weights=[.9, .1], flip_y=.02, random_state=SEED)
ids = np.arange(len(y))
train_ids, held_ids = train_test_split(ids, test_size=.4, stratify=y, random_state=SEED)
val_ids, test_ids = train_test_split(held_ids, test_size=.5, stratify=y[held_ids], random_state=SEED+1)
assert set(train_ids).isdisjoint(val_ids) and set(held_ids).isdisjoint(train_ids)
models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'knn': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    'svm_rbf': make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED)),
    'tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=SEED),
    'random_forest': RandomForestClassifier(n_estimators=80, min_samples_leaf=3,
                                             random_state=SEED, n_jobs=1),
    'boosting': HistGradientBoostingClassifier(max_iter=80, max_leaf_nodes=15, random_state=SEED),
}
rows = []
for name, model in models.items():
    begin = perf_counter(); model.fit(X[train_ids], y[train_ids])
    rows.append({'model': name, 'fit_seconds': perf_counter()-begin,
                 'val_ap': average_precision_score(y[val_ids], model.predict_proba(X[val_ids])[:, 1])})
selection = pd.DataFrame(rows).sort_values('val_ap', ascending=False)
display(selection)
selected_name = str(selection.iloc[0]['model']); selected = models[selected_name]


## 3. Validation 选 threshold；只评价选定模型的 test
这里允许 validation 同时选择模型和阈值，因此最终必须用独立 test。


In [ ]:
val_prob = selected.predict_proba(X[val_ids])[:, 1]
thresholds = np.linspace(.05, .95, 91)
val_f1 = [f1_score(y[val_ids], val_prob >= t, zero_division=0) for t in thresholds]
threshold = float(thresholds[int(np.argmax(val_f1))])
test_prob = selected.predict_proba(X[test_ids])[:, 1]
prediction = test_prob >= threshold
test_y = y[test_ids]
results = {'seed': SEED, 'selected_model': selected_name, 'validation_selected_threshold': threshold,
    'test_n': len(test_ids), 'test_prevalence': float(test_y.mean()),
    'test_accuracy': float(accuracy_score(test_y, prediction)),
    'test_precision': float(precision_score(test_y, prediction, zero_division=0)),
    'test_recall': float(recall_score(test_y, prediction, zero_division=0)),
    'test_f1': float(f1_score(test_y, prediction, zero_division=0)),
    'test_ap': float(average_precision_score(test_y, test_prob)),
    'test_roc_auc': float(roc_auc_score(test_y, test_prob)),
    'test_brier': float(brier_score_loss(test_y, test_prob)),
    'always_negative_accuracy': float((test_y == 0).mean()),
    'confusion_matrix': confusion_matrix(test_y, prediction).tolist()}
print(json.dumps(results, indent=2))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
precision, recall, _ = precision_recall_curve(test_y, test_prob)
axes[0].plot(recall, precision)
axes[0].axhline(test_y.mean(), color='black', linestyle='--')
axes[0].set(xlabel='recall', ylabel='precision', title='Test PR curve')
fpr, tpr, _ = roc_curve(test_y, test_prob)
axes[1].plot(fpr, tpr); axes[1].plot([0, 1], [0, 1], '--', color='black')
axes[1].set(xlabel='FPR', ylabel='TPR', title='Test ROC curve')
fraction, mean_prob = calibration_curve(test_y, test_prob, n_bins=6)
axes[2].plot(mean_prob, fraction, 'o-'); axes[2].plot([0,1],[0,1],'--',color='black')
axes[2].set(xlabel='mean probability', ylabel='positive fraction', title='Reliability diagram')
plt.tight_layout(); plt.show()


## 4. 固定规则的 bootstrap 区间
此合成数据中每行独立，按行 bootstrap；真实机器人重复 rollout/episode 内的 frames 不满足这个假设。


In [ ]:
bootstrap_rng = np.random.default_rng(SEED+100)
scores = []
for _ in range(400):
    sample = bootstrap_rng.integers(0, len(test_y), size=len(test_y))
    scores.append(f1_score(test_y[sample], prediction[sample], zero_division=0))
results['test_f1_bootstrap_95_percentile_interval'] = np.quantile(scores, [.025, .975]).tolist()
print('F1 percentile interval:', results['test_f1_bootstrap_95_percentile_interval'])


## 5. 独立反例：frame split 记住 group ID
下面不是重新调上面的 test。每组 label 随机、组内共享 ID；新组没有可泛化信号，1NN 可记住见过的组。


In [ ]:
group_rng = np.random.default_rng(SEED+10)
n_groups, per_group = 100, 8
groups = np.repeat(np.arange(n_groups), per_group)
group_labels = group_rng.integers(0, 2, size=n_groups)
gy = group_labels[groups]
gx = np.eye(n_groups)[groups]
frame_train, frame_eval = train_test_split(np.arange(len(gy)), test_size=.3, random_state=SEED)
group_train, group_eval = next(GroupShuffleSplit(n_splits=1, test_size=.3, random_state=SEED).split(gx, gy, groups))
leakage_rows = []
for name, tr, ev in [('frame split', frame_train, frame_eval), ('group split', group_train, group_eval)]:
    memorizer = KNeighborsClassifier(n_neighbors=1).fit(gx[tr], gy[tr])
    leakage_rows.append({'split': name, 'accuracy': float(accuracy_score(gy[ev], memorizer.predict(gx[ev]))),
                         'overlap_groups': len(set(groups[tr]) & set(groups[ev]))})
results['group_identity_counterexample'] = leakage_rows
display(pd.DataFrame(leakage_rows))


## 面试追问与实验记录
1. 为什么 always-negative accuracy 可能高，但 recall 为零？
2. AP、ROC-AUC、threshold F1 与 Brier 各看什么？
3. group split 的独立单位是什么？
4. 小样本 bootstrap 的区间为什么不是普遍保证？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
